# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames the Refresh / Content Opportunity Scoring lane before modeling. The starter CSV is a retrospective teaching slice; the framing below distinguishes its proxy label from a future outcome that would be needed for a leakage-safe production claim.

## 1. My lane as an ML task (type)

My lane is **ranking / scoring**. The editor has limited review time and needs to know which pages to inspect first, so the useful output is an ordered queue, not just a yes/no prediction. The person acting on it is a content editor or SEO lead; the action is to review the highest-priority pages for a possible refresh. This is decision support, not an automatic instruction to edit.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path('../../data/raw/content_refresh_anonymized.csv')
df = pd.read_csv(data_path)
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype('int8')

print(f"Rows (content items): {len(df):,}")
print(f"Clients available for grouped validation: {df['client_id'].nunique():,}")
print('Output: an ordered page-review queue for an editor, not an automated refresh decision.')

Rows (content items): 30,000
Clients available for grouped validation: 32
Output: an ordered page-review queue for an editor, not an automated refresh decision.


## 2. Target or proxy

For an initial exercise, the proxy target is `is_declining_label = 1` when `trend_direction == 'down'`, otherwise `0`. It marks an observed retrospective trend: impressions in the most recent 30 days were more than 20% below the previous 30 days. It is not a future outcome, so it does not yet support a claim that the score predicts what will happen next.

Important limitation: the starter snapshot contains the same trend windows used to define this proxy. `trend_direction` and `trend_pct` are label sources and must never be features; other same-window inputs can also leak the outcome. A valid future-risk model needs a feature cutoff before a strictly later outcome window (best built from the daily warehouse).

In [2]:
label_counts = df['is_declining_label'].value_counts().sort_index()
print(f"Proxy label definition: trend_direction == 'down'")
print(f"Declining pages: {int(label_counts.loc[1]):,} / {len(df):,} ({df['is_declining_label'].mean():.1%})")
print(f"Non-declining pages: {int(label_counts.loc[0]):,} / {len(df):,} ({1 - df['is_declining_label'].mean():.1%})")
assert df['is_declining_label'].isin([0, 1]).all()
assert {'trend_direction', 'trend_pct'}.issubset(df.columns)

Proxy label definition: trend_direction == 'down'
Declining pages: 16,262 / 30,000 (54.2%)
Non-declining pages: 13,738 / 30,000 (45.8%)


## 3. Success metric

The primary metric is **Precision@50**: among the 50 pages at the top of the queue, what fraction have the observed decline label? It matches the editor's bounded review capacity better than overall accuracy. The full-slice prevalence is 54.2%, so a random top-50 list would average about 27 declining pages; a useful model must beat the prevalence of its held-out clients and a transparent rule baseline on client-held-out validation. That 54.2% is context, not a held-out result or a model score.

In [3]:
k = 50
base_rate = df['is_declining_label'].mean()
print(f"Full-slice reference prevalence: {base_rate:.1%}")
print(f"Expected declining pages in a random top-{k} list: {k * base_rate:.1f} (reference only)")
print(f"Success criterion: held-out-client Precision@{k} > that holdout's prevalence and the rule baseline.")

Full-slice reference prevalence: 54.2%
Expected declining pages in a random top-50 list: 27.1 (reference only)
Success criterion: held-out-client Precision@50 > that holdout's prevalence and the rule baseline.


## 4. The unit of analysis, as a real dataframe

The unit is **one pseudonymized content item (page)**. Each row represents one page from the starter slice; the sample below shows a few observed signals and the proxy target. IDs are kept for grouping and validation only, not as model inputs.

In [4]:
unit_columns = [
    'content_type',
    'impressions_90d',
    'avg_position',
    'content_age_days',
    'is_declining_label',
]
page_rows = df[unit_columns].head(8).copy()
print(f"One row = one page; full dataframe shape = {df.shape}")
print(f"Unique content items = {df['content_id'].nunique():,}; rows = {len(df):,}")
display(page_rows)
assert df['content_id'].is_unique
assert page_rows['is_declining_label'].isin([0, 1]).all()

One row = one page; full dataframe shape = (30000, 45)
Unique content items = 30,000; rows = 30,000


,content_type,impressions_90d,avg_position,content_age_days,is_declining_label
0,keyword article,3803,10.6,187,1
1,keyword article,15320,20.3,445,1
2,keyword article,12581,36.5,141,1
3,keyword article,11751,6.2,463,0
4,keyword article,19140,44.0,263,1
5,keyword article,3970,8.5,147,1
6,keyword article,20,7.0,90,1
7,keyword article,1724,21.2,445,0


## 5. Why ML beats a fixed rule here

A single threshold (for example, refresh every page with low impressions) ignores context: pages differ in age, content type, search visibility, engagement, and available measurement, and those signals can interact. A scoring approach can combine multiple eligible pre-decision signals into a ranked review queue. But ML has to earn its place: I will compare it with a simple rule baseline and keep it only if client-held-out Precision@50 improves enough to justify the extra complexity. If the rule performs as well, the rule is the better tool. The current snapshot is for framing only; its overlapping outcome windows mean I must not use it to claim future prediction.

In [5]:
# Check the data conditions that matter to this framing: row grain and grouped evaluation.
assert df['content_id'].is_unique, 'Expected one row per content item.'
assert df['client_id'].nunique() > 1, 'Need multiple clients for client-held-out validation.'
assert len(df) == 30_000
print('Framing checks passed: unique page rows, multiple clients, and an observed binary proxy label.')
print('Next: define a time-safe feature cutoff and future outcome window before training or making predictive claims.')

Framing checks passed: unique page rows, multiple clients, and an observed binary proxy label.
Next: define a time-safe feature cutoff and future outcome window before training or making predictive claims.


## 6. Self-check

- [x] Named the lane, task type, target/proxy, metric, decision, and reason to test ML against a rule
- [x] Loaded the starter slice and showed a real dataframe with one row per page
- [x] Clearly labeled the retrospective proxy and its leakage limitation
- [x] The notebook runs top to bottom
- [x] No client names, raw URLs, or private queries are included
- [x] Commit this notebook to the repository before submitting the repo URL